In [1]:
from scipy.io import loadmat, whosmat
import numpy as np
import pandas as pd
import xarray as xr
import warnings
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from bluemath_tk.core.plotting.utils import join_colormaps

from bluemath_tk.core.plotting.base_plotting import DefaultStaticPlotting
plot = DefaultStaticPlotting()

In [2]:
def mat_to_xr_dataset(path, variables=None, time_range=None):
    """
    Convert a MATLAB file to xarray.Dataset, loading only requested data.
    
    Parameters
    ----------
    path : str
        Path to .mat file
    variables : list of str, optional
        Variable names to load (e.g., ['Hsig', 'Windv_x']). 
        If None, loads all variables.
    time_range : tuple of (start, end), optional
        Only load timesteps within this range.
        e.g., ('2000-01-01', '2000-01-02')
    
    Returns
    -------
    xr.Dataset
    """
    mat_info = whosmat(path)
    all_keys = [name for name, shape, dtype in mat_info]

    vars_with_keys = {}
    
    for key in all_keys:
        if key.startswith(("Xp", "Yp", "__")):
            continue
        parts = key.rsplit("_", 2)
        if len(parts) >= 3:
            varname = "_".join(parts[:-2])
            timestamp_str = f"{parts[-2]}_{parts[-1]}"
            try:
                timestamp = pd.to_datetime(timestamp_str, format="%Y%m%d_%H%M%S")
                vars_with_keys.setdefault(varname, []).append((key, timestamp))
            except ValueError:
                continue

    if variables is not None:
        vars_with_keys = {k: v for k, v in vars_with_keys.items() if k in variables}
    
    if not vars_with_keys:
        raise ValueError(f"No matching variables found. Available: {list(vars_with_keys.keys())}")

    if time_range is not None:
        t_start, t_end = pd.to_datetime(time_range[0]), pd.to_datetime(time_range[1])
        for varname in vars_with_keys:
            vars_with_keys[varname] = [
                (k, t) for k, t in vars_with_keys[varname] 
                if t_start <= t <= t_end
            ]

    keys_to_load = {"Xp", "Yp"}
    for varname, key_times in vars_with_keys.items():
        keys_to_load.update(k for k, t in key_times)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Duplicate variable name")
        mat = loadmat(path, variable_names=list(keys_to_load))
    
    lon = mat["Xp"].ravel().astype(np.float32)
    lat = mat["Yp"].ravel().astype(np.float32)

    ds_vars = {}

    for varname, key_times in vars_with_keys.items():
        key_times.sort(key=lambda x: x[1])  # Sort by timestamp
        
        n_times = len(key_times)
        n_points = lon.size
        data_array = np.empty((n_times, n_points), dtype=np.float32)
        
        for i, (key, _) in enumerate(key_times):
            data_array[i] = mat[key].ravel()
        
        time_index = pd.DatetimeIndex([t for _, t in key_times])
        
        unique_times, time_indices = np.unique(time_index, return_index=True)
        data_array = data_array[time_indices]
        time_index = pd.DatetimeIndex(unique_times)
        
        ds_vars[varname] = (("time", "points"), data_array)
    
    return xr.Dataset(
        data_vars=ds_vars,
        coords={
            "time": time_index,
            "lon": ("points", lon),
            "lat": ("points", lat),
        },
    )


def read_adcirc_grd(grd_file: str):

    with open(grd_file, "r") as f:
        _header0 = f.readline()
        header1 = f.readline()
        header_nums = list(map(float, header1.split()))
        nelmts = int(header_nums[0])
        nnodes = int(header_nums[1])

        Nodes = np.loadtxt(f, max_rows=nnodes)
        Elmts = np.loadtxt(f, max_rows=nelmts) - 1
        lines = f.readlines()

    return Nodes, Elmts, lines

In [3]:
Nodes_calc, Elmts_calc, lines_calc = read_adcirc_grd("templates/fort.14")

triangles = Elmts_calc[:, 2:5].astype(int)

In [6]:
ncols, nrows = 8, 8

selected_cases = 13139

cmap = join_colormaps(
    cmap1="viridis",
    cmap2="plasma_r",
    name="wind_partition_cmap",
    range1=(0.2, 1.0),
    range2=(0.05, 0.8),
)

plot = DefaultStaticPlotting()

area = [
    Nodes_calc[:, 1].min(),
    Nodes_calc[:, 1].max(),
    Nodes_calc[:, 2].min(),
    Nodes_calc[:, 2].max(),
]

lon_range = area[1] - area[0]
lat_range = area[3] - area[2]
aspect_ratio = lon_range / lat_range

panel_height = 4
panel_width = panel_height * aspect_ratio

# Espace fixe en pouces (indépendant du nombre de panneaux)
title_height = 1.5   # pouces pour le titre
cbar_height = 1.2    # pouces pour la colorbar

fig_temp, ax_temp = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
ax_temp.set_extent(area, crs=ccrs.PlateCarree())
plot.plot_satellite(ax=ax_temp, source="arcgis", area=area)

satellite_img = None
satellite_extent = None
for child in ax_temp.get_children():
    if hasattr(child, 'get_array') and hasattr(child, 'get_extent'):
        arr = child.get_array()
        if arr is not None and arr.size > 100:
            satellite_img = arr
            satellite_extent = child.get_extent()
            break
plt.close(fig_temp)

figsize = (8, 8)


Using Esri World Imagery (ArcGIS):
- Free for public and non-commercial use
- Commercial use allowed under Esri terms of service
- Attribution required: 'Tiles © Esri — Sources: Esri, Earthstar Geographics, CNES/Airbus DS, USDA, USGS, AeroGRID, IGN, and the GIS User Community'
- Max zoom ~19


In [56]:
from scipy.interpolate import griddata

n_lon, n_lat = 35, 30
lon_reg = np.linspace(area[0], area[1], n_lon)
lat_reg = np.linspace(area[2], area[3], n_lat)
Lon_grid, Lat_grid = np.meshgrid(lon_reg, lat_reg)

points_mesh = Nodes_calc[:, [1, 2]]

swan_output = f"cases/metamodel/{selected_cases}/output.mat"
ds = mat_to_xr_dataset(
    swan_output,
    variables=["Hsig", "Dir", "TDir", "TPsmoo", "Tm01", "Windv_x", "Windv_y"],
)

Hsmin, Hsmax = ds["Hsig"].min().item(), ds["Hsig"].max().item()
W = np.sqrt(ds["Windv_x"] ** 2 + ds["Windv_y"] ** 2)
Dir_wind = np.arctan2(ds["Windv_y"], ds["Windv_x"])
Wmin, Wmax = W.min().item(), W.max().item()

# Sous-ensemble des temps (à partir de l'index 12)
time_slice = ds.time.values[11:-20]

for j, t in enumerate(time_slice):
    fig, (axes_1, axes_2) = plt.subplots(
        ncols=2,
        subplot_kw={"projection": ccrs.PlateCarree()},
        figsize=(15, 8),
        constrained_layout=True,
    )

    axes_1.set_extent(area, crs=ccrs.PlateCarree())
    axes_1.imshow(
        satellite_img,
        extent=satellite_extent,
        transform=ccrs.PlateCarree(),
        origin="lower",
        zorder=0,
    )

    facecolor_Hs = np.mean(ds.Hsig.sel(time=t).values[triangles], axis=1)
    pcm1 = axes_1.tripcolor(
        Nodes_calc[:, 1],
        Nodes_calc[:, 2],
        triangles,
        facecolor_Hs,
        shading="flat",
        cmap="jet",
        vmin=Hsmin,
        vmax=Hsmax,
        zorder=1,
    )

    dir_deg = ds.Dir.sel(time=t).values

    angle = np.deg2rad((270 - ds.Dir.sel(time=t).values) % 360)

    u_wave = np.cos(angle)
    v_wave = np.sin(angle)
    u_wave_grid = griddata(points_mesh, u_wave, (Lon_grid, Lat_grid), method="linear")
    v_wave_grid = griddata(points_mesh, v_wave, (Lon_grid, Lat_grid), method="linear")
    mask = np.isnan(u_wave_grid)
    u_wave_grid = np.ma.masked_where(mask, u_wave_grid)
    v_wave_grid = np.ma.masked_where(mask, v_wave_grid)

    axes_1.quiver(
        Lon_grid,
        Lat_grid,
        u_wave_grid,
        v_wave_grid,
        scale=50,
        width=0.002,
        color="black",
        zorder=2,
        transform=ccrs.PlateCarree(),
    )
    fig.colorbar(pcm1, ax=axes_1, orientation="horizontal", pad=0.05, label="Significant Wave Height (m)")

    axes_2.set_extent(area, crs=ccrs.PlateCarree())
    axes_2.imshow(
        satellite_img,
        extent=satellite_extent,
        transform=ccrs.PlateCarree(),
        origin="lower",
        zorder=0,
    )

    facecolor_W = np.mean(W.sel(time=t).values[triangles], axis=1)
    pcm2 = axes_2.tripcolor(
        Nodes_calc[:, 1],
        Nodes_calc[:, 2],
        triangles,
        facecolor_W,
        shading="flat",
        cmap=cmap,  # à définir si besoin, ex. cmap = "viridis"
        vmin=Wmin,
        vmax=Wmax,
        zorder=1,
    )

    quiver_wind = Dir_wind.sel(time=t)
    u_wind = np.cos(quiver_wind.values)
    v_wind = np.sin(quiver_wind.values)
    u_wind_grid = griddata(points_mesh, u_wind, (Lon_grid, Lat_grid), method="linear")
    v_wind_grid = griddata(points_mesh, v_wind, (Lon_grid, Lat_grid), method="linear")
    mask_w = np.isnan(u_wind_grid)
    u_wind_grid = np.ma.masked_where(mask_w, u_wind_grid)
    v_wind_grid = np.ma.masked_where(mask_w, v_wind_grid)

    axes_2.quiver(
        Lon_grid,
        Lat_grid,
        u_wind_grid,
        v_wind_grid,
        scale=50,
        width=0.002,
        color="black",
        zorder=2,
        transform=ccrs.PlateCarree(),
    )
    fig.colorbar(pcm2, ax=axes_2, orientation="horizontal", pad=0.05, label="Wind Speed (m/s)")

    time_label = pd.Timestamp(t).strftime("%Y-%m-%d %H:%M") if np.issubdtype(type(t), np.datetime64) else str(t)
    plt.suptitle(f"Time: {time_label}", fontsize=16)
    axes_1.set_title("Significant Wave Height with Wave Direction")
    axes_2.set_title("Wind Speed with Wind Direction")

    fig.savefig(f"figures/10_fer_plots/fer_plot_{j:03d}.png", dpi=150)
    plt.close()

In [63]:
import subprocess
import glob

img_pattern = "figures/10_fer_plots/fer_plot_*.png"
out_video = "figures/10_fer_plots/fer_video.mp4"
fps = 3

imgs = sorted(glob.glob(img_pattern))
if not imgs:
    raise SystemExit(f"Aucune image trouvée pour {img_pattern}")

subprocess.run(
    [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", "figures/10_fer_plots/fer_plot_%03d.png",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        out_video,
    ],
    check=True,
)
print(f"Vidéo sauvegardée : {out_video}")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 13.3.0 (conda-forge gcc 13.3.0-2)
  configuration: --prefix=/nfs/home/geocean/faugeree/miniforge3/envs/work --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1748704137276/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1748704137276/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1748704137276/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1748704137276/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-alsa --enable-libpulse --enable-vaapi --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 -

Vidéo sauvegardée : figures/10_fer_plots/fer_video.mp4


[out#0/mp4 @ 0x55972802e640] video:3026KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.049609%
frame=   51 fps=0.0 q=-1.0 Lsize=    3027KiB time=00:00:16.33 bitrate=1518.3kbits/s speed=25.6x    
[libx264 @ 0x55972802f7c0] frame I:1     Avg QP:11.85  size:436956
[libx264 @ 0x55972802f7c0] frame P:13    Avg QP:15.92  size: 97265
[libx264 @ 0x55972802f7c0] frame B:37    Avg QP:19.21  size: 37734
[libx264 @ 0x55972802f7c0] consecutive B-frames:  2.0%  3.9%  0.0% 94.1%
[libx264 @ 0x55972802f7c0] mb I  I16..4: 31.1% 20.9% 47.9%
[libx264 @ 0x55972802f7c0] mb P  I16..4:  3.2%  3.8% 10.0%  P16..4:  6.0%  4.3%  3.7%  0.0%  0.0%    skip:68.9%
[libx264 @ 0x55972802f7c0] mb B  I16..4:  0.9%  0.8%  1.0%  B16..8:  7.5%  4.7%  2.5%  direct: 6.5%  skip:76.1%  L0:24.6% L1:56.8% BI:18.6%
[libx264 @ 0x55972802f7c0] 8x8 transform intra:23.3% inter:40.5%
[libx264 @ 0x55972802f7c0] coded y,uvDC,uvAC intra: 46.9% 83.2% 76.7% inter: 7.8% 16.8% 10.3%
[libx264 @ 0x55972802f